LOADING CSV FILE INTO PANDAS

In [6]:
import pandas as pd

df = pd.read_csv("utility_outage_data.csv")

df.head()

,outage_id,region,start_time,end_time,affected_customers,cause
0,OUT00001,Central,2024-02-25 10:29:07,2024-02-25 16:07:07,65,Storm
1,OUT00002,East,2025-09-17 07:00:48,2025-09-17 15:04:48,21,Animal Contact
2,OUT00003,South,2024-01-05 03:25:57,2024-01-05 04:33:57,91,Grid Overload
3,OUT00004,North,2025-12-10 10:59:58,2025-12-10 17:34:58,67,Storm
4,OUT00005,Central,2025-04-18 20:16:52,2025-04-19 06:46:52,72,Grid Overload


CHECKING FOR DATA VALIDITY AND DATAFRAME INFORMATION

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   outage_id           500 non-null    str  
 1   region              500 non-null    str  
 2   start_time          500 non-null    str  
 3   end_time            500 non-null    str  
 4   affected_customers  500 non-null    int64
 5   cause               500 non-null    str  
dtypes: int64(1), str(5)
memory usage: 55.3 KB


TRANFORMING DATAFRAME COLUMN

In [8]:
df.columns = [c.upper() for c in df.columns]

df.head()

,OUTAGE_ID,REGION,START_TIME,END_TIME,AFFECTED_CUSTOMERS,CAUSE
0,OUT00001,Central,2024-02-25 10:29:07,2024-02-25 16:07:07,65,Storm
1,OUT00002,East,2025-09-17 07:00:48,2025-09-17 15:04:48,21,Animal Contact
2,OUT00003,South,2024-01-05 03:25:57,2024-01-05 04:33:57,91,Grid Overload
3,OUT00004,North,2025-12-10 10:59:58,2025-12-10 17:34:58,67,Storm
4,OUT00005,Central,2025-04-18 20:16:52,2025-04-19 06:46:52,72,Grid Overload


ESTABLISH CONNECTION TO SNOWFLAKE

In [9]:
import snowflake.connector
from dotenv import load_dotenv
import os

load_dotenv()

conn = None

try:
    conn = snowflake.connector.connect(
        account = os.getenv("SNOWFLAKE_ACCOUNT"),
        user = os.getenv("SNOWFLAKE_USER"),
        role = os.getenv("SNOWFLAKE_ROLE"),
        warehouse = os.getenv("SNOWFLAKE_WAREHOUSE"),
        database = os.getenv("SNOWFLAKE_DATABASE"),
        schema = os.getenv("SNOWFLAKE_SCHEMA"),
        password = os.getenv("SNOWFLAKE_PASSWORD")
    )
    print("Successfully connected to Snowflake!")
except Exception as e:
    print(f"Error: {e}")


Successfully connected to Snowflake!


TESTING CONNECTION TO SNOWFLAKE

In [10]:
try:
    cur = conn.cursor() # type: ignore
    cur.execute("SELECT 1")
    print("Connection OK")
except Exception as e:
    print("Connection failed:", e)

Connection OK


LOADING TO SNOWFLAKE

In [11]:
from snowflake.connector.pandas_tools import write_pandas
import traceback


if conn:
    try:
        success, nchunks, nrows, _ = write_pandas(
        conn,
        df,                      # your pandas DataFrame
        table_name="UTILITY_OUTAGE_DATA",  # target table in Snowflake
        )
        print(success, nrows)
        print(f"Successfully loaded {nrows} rows into Snowflake.")
    except Exception as e:
        print(f"Error: {e}")
        traceback.print_exc()


True 500
Successfully loaded 500 rows into Snowflake.


CLOSE CONNECTION

In [12]:
try:
    conn.close() # type: ignore
    print("Connection closed.")
except Exception as e:
    print(f"Error closing connection: {e}")

Connection closed.
